In [6]:
import pandas as pd
import numpy as np

def calculate_goals(
    sales_file_path,
    sales_sheet,
    mapping_sheet,
    r12_months,
    products,
    product_weights,
    cap_min_terr_growth,
    cap_max_terr_growth,
    national_goal
):
    # Load data
    df_sales = pd.read_excel(sales_file_path, sheet_name=sales_sheet)
    df_mapping = pd.read_excel(sales_file_path, sheet_name=mapping_sheet, skiprows=5, usecols="C:D")
    df_mapping.drop_duplicates(inplace=True)

    # Merge
    df = df_sales.merge(df_mapping, how="left", left_on="Territory", right_on="TS Territory")
    df.drop("TS Territory", axis=1, inplace=True)

    # Compute R12 Totals
    for product in products:
        df[f"{product} R12 Total"] = df[[f"{product} R12 {month}" for month in r12_months]].sum(axis=1)

    # Keep required columns
    selected_columns = ["Region", "Territory", "Territory Name"] + [f"{p} R12 Total" for p in products]
    df = df[selected_columns]

    # Baseline Goal
    df['Baseline Goal'] = np.round(df[f'{products[0]} R12 Total'] / 2)
    baseline_goal = df['Baseline Goal'].sum()
    print("Baseline Goal:", baseline_goal)

    # Market Index
    for product in products[1:]:
        df[f"{product} Index"] = np.round(
            (df[f"{product} R12 Total"] / df[f"{product} R12 Total"].sum()) * 100, 1
        )

    df['Market Index'] = sum(df[f"{p} Index"] * product_weights[p] for p in products[1:])
    df['Market Index'] = np.round(df['Market Index'] / 100, 1)

    # Growth and Preliminary Goal
    df["Growth/Potential Goal"] = np.round(abs(baseline_goal - national_goal) * df['Market Index'] / 100, 1)
    df["Preliminary Goal"] = df['Baseline Goal'] + df["Growth/Potential Goal"]
    df['Adjusted Preliminary Goal'] = np.round((df["Preliminary Goal"] / df["Preliminary Goal"].sum()) * national_goal, 1)
    df['Growth%'] = np.round((df['Adjusted Preliminary Goal'] / df['Baseline Goal'] - 1) * 100, 1)

    national_growth = np.round(((baseline_goal / national_goal) - 1) * 100, 1)
    print("National Growth:", national_growth)

    # Growth Capping
    df['Capped Flag'] = np.where(
        df['Growth%'] < cap_min_terr_growth, 1,
        np.where(df['Growth%'] > cap_max_terr_growth, 2, 0)
    )
    df['Capped Growth%'] = df['Growth%'].clip(lower=cap_min_terr_growth, upper=cap_max_terr_growth)

    # Initial Capped Goal
    df['Initial Capped Goal'] = np.where(
        df['Baseline Goal'] == 0,
        df['Growth%'],
        (1 + df['Capped Growth%'] / 100) * df['Baseline Goal']
    )

    # Spare growth logic
    total_diff = max(0, df['Adjusted Preliminary Goal'].sum() - df['Initial Capped Goal'].sum())

    df['Spare growth'] = np.where(
        total_diff > 0,
        (cap_max_terr_growth / 100 - df['Capped Growth%'] / 100) * df['Baseline Goal'],
        (df['Capped Growth%'] / 100 - cap_min_terr_growth / 100) * df['Baseline Goal']
    )

    # Final Goal calculation
    total_spare_growth = df['Spare growth'].sum()
    df["% of Nation"] = df['Spare growth'] / total_spare_growth * 100
    df["Goal Difference (Delta)"] = df["% of Nation"] * total_diff
    df["Final Goal (cartons)"] = np.round(df["Goal Difference (Delta)"] + df['Initial Capped Goal'], 0)

    return df


In [ ]:
sales_file_path = r"C:\IC Agents\crewai\icagents\Goal Setting sanitized Data.xlsx"
sales_sheet = "Input Sales"
mapping_sheet = "1c. Inputs - Mappings"
r12_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
products = ["Product 1", "Product 2", "Product 3"]
product_weights = {"Product 2": 60, "Product 3": 40}
cap_min_terr_growth = -33.9
cap_max_terr_growth = 33.9
national_goal = 5501

result_df = calculate_goals(
    sales_file_path,
    sales_sheet,
    mapping_sheet,
    r12_months,
    products,
    product_weights,
    cap_min_terr_growth,
    cap_max_terr_growth,
    national_goal
)

result_df.head()


Baseline Goal: 5536.0
National Growth: 0.6


,Region,Territory,Territory Name,Product 1 R12 Total,Product 2 R12 Total,Product 3 R12 Total,Baseline Goal,Product 2 Index,Product 3 Index,Market Index,...,Preliminary Goal,Adjusted Preliminary Goal,Growth%,Capped Flag,Capped Growth%,Initial Capped Goal,Spare growth,% of Nation,Goal Difference (Delta),Final Goal (cartons)
0,101,101001,East Coast,481,341,1361,240.0,2.6,3.1,2.8,...,241.0,238.0,-0.8,0,-0.8,238.080,79.440,4.312331,0.0,238.0
1,101,101002,Northeast Hub,443,397,2066,222.0,3.1,4.7,3.7,...,223.3,220.5,-0.7,0,-0.7,220.446,73.704,4.000958,0.0,220.0
2,101,101003,Metro East,456,208,968,228.0,1.6,2.2,1.8,...,228.6,225.7,-1.0,0,-1.0,225.720,75.012,4.071961,0.0,226.0
3,101,101004,Atlantic Zone,641,788,3283,320.0,6.1,7.4,6.6,...,322.3,318.3,-0.5,0,-0.5,318.400,106.880,5.801888,0.0,318.0
4,101,101005,Capital Region,602,503,1966,301.0,3.9,4.4,4.1,...,302.4,298.6,-0.8,0,-0.8,298.592,99.631,5.408382,0.0,299.0


In [10]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\deban\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np


In [34]:
import pandas as pd


df = pd.read_excel('C:\IC Agents\crewai\icagents\Goal Setting sanitized Data.xlsx', sheet_name='Input Sales')  # Assuming header starts at row 4

# Clean column names
df.columns = df.columns
columns_to_sum = [
    "Product 1 R12 Sep-13",
    "Product 1 R12 Aug-13",
    "Product 1 R12 Jul-13"
]

df["Product 1 Q2"] = df[columns_to_sum].sum(axis=1)
df=df[["Territory", "Territory Name", "Product 1 Q2"]]

print(df.columns)
df.head()


Index(['Territory', 'Territory Name', 'Product 1 Q2'], dtype='object')


<>:4: SyntaxWarning: invalid escape sequence '\I'
<>:4: SyntaxWarning: invalid escape sequence '\I'
C:\Users\deban\AppData\Local\Temp\ipykernel_37568\3737124266.py:4: SyntaxWarning: invalid escape sequence '\I'
  df = pd.read_excel('C:\IC Agents\crewai\icagents\Goal Setting sanitized Data.xlsx', sheet_name='Input Sales')  # Assuming header starts at row 4


,Territory,Territory Name,Product 1 Q2
0,101001,East Coast,239
1,101002,Northeast Hub,224
2,101003,Metro East,220
3,101004,Atlantic Zone,339
4,101005,Capital Region,289


In [23]:
import pandas as pd
def detect_product1_anomalies_dynamic(sales_df: pd.DataFrame, z_thresh: float = -2.0) -> pd.DataFrame:
    """
    Detects anomalies in Product 1 sales (Jan–Jun) using z-score thresholding.

    Parameters:
    - sales_df: pd.DataFrame with Product 1 R12 Jan–Jun columns
    - z_thresh: float, the z-score below which a value is flagged as anomaly (default -2.0)

    Returns:
    - pd.DataFrame with anomalies including z-score
    """
    target_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
    product1_cols = [f"Product 1 R12 {m}" for m in target_months]

    # Check for missing columns
    missing = [col for col in product1_cols if col not in sales_df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
    # Melt to long format
    df_long = sales_df.melt(
        id_vars=["Territory", "Territory Name"],
        value_vars=product1_cols,
        var_name="Month",
        value_name="Sales"
    )
    df_long["Month"] = df_long["Month"].str.extract(r'R12 (\w+)', expand=False)

    # Compute z-scores by month
    stats = df_long.groupby("Month")["Sales"].agg(["mean", "std"]).reset_index()
    df_merged = df_long.merge(stats, on="Month")
    df_merged["z_score"] = (df_merged["Sales"] - df_merged["mean"]) / df_merged["std"]

    # Flag anomalies below z-score threshold
    anomalies = df_merged[df_merged["z_score"] < z_thresh].copy()
    anomalies = anomalies[["Territory", "Territory Name", "Month", "Sales", "z_score"]].reset_index(drop=True)
    
    return anomalies

def cross_verify_anomalies_with_fema(anomalies_df: pd.DataFrame, fema_df: pd.DataFrame) -> pd.DataFrame:
    """
    Matches anomalies to FEMA disasters by comparing territory and month.

    Parameters:
    - anomalies_df: DataFrame with at least ['Territory', 'Territory Name', 'Month', 'Sales', 'z_score']
    - fema_df: FEMA DataFrame with at least ['TS Territory ID', 'DeclarationDate', 'IncidentType', 'DeclarationTitle']

    Returns:
    - pd.DataFrame with FEMA match results and a 'Disaster Match' flag
    """

    # 1. Normalize month format in anomalies
    month_map = {
        "Jan": "January", "Feb": "February", "Mar": "March",
        "Apr": "April", "May": "May", "Jun": "June"
    }
    anomalies_df = anomalies_df.copy()
    anomalies_df["IncidentMonth"] = anomalies_df["Month"].map(month_map)

    # 2. Prepare FEMA data
    fema_df = fema_df.copy()
    fema_df["IncidentBeginDate"] = pd.to_datetime(fema_df["IncidentBeginDate"], errors="coerce")
    fema_df["IncidentMonth"] = fema_df["IncidentBeginDate"].dt.strftime("%B")
    fema_df["TS Territory ID"] = pd.to_numeric(fema_df["TS Territory ID"], errors="coerce").astype("Int64")

    fema_subset = fema_df[[
        "TS Territory ID", "IncidentBeginDate","IncidentEndDate","IncidentMonth", "IncidentType", "DeclarationTitle"
    ]].drop_duplicates()
    # 3. Merge anomalies with FEMA data
    merged = anomalies_df.merge(
        fema_subset,
        how="left",
        left_on=["Territory", "IncidentMonth"],  # from anomalies_df
        right_on=["TS Territory ID", "IncidentMonth"]  # from FEMA
    )

    # 4. Flag match
    merged["Disaster Match"] = ~merged["IncidentType"].isna()
    merged.drop_duplicates(inplace=True)

    # 5. Return extended anomalies
    return merged
def format_disaster_impact_summary_html(verified_anomalies_df: pd.DataFrame, sales_df: pd.DataFrame) -> str:
    """
    Generate a detailed HTML summary of anomalies with disaster impact,
    comparing each anomalous month to other months from the same territory using sales_df.
    """

    disaster_rows = verified_anomalies_df[verified_anomalies_df["Disaster Match"] == True]

    if disaster_rows.empty:
        return """
        <div style='padding: 1rem; background-color: #eafbea; border-radius: 10px;'>
            ✅ No disasters were detected that impacted your sales data. Goal calculation proceeded normally.
        </div>
        """

    # Prepare long-form version of sales_df for month-wise analysis
    target_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
    product1_cols = [f"Product 1 R12 {m}" for m in target_months]

    df_long = sales_df.melt(
        id_vars=["Territory", "Territory Name"],
        value_vars=product1_cols,
        var_name="Month",
        value_name="Sales"
    )
    df_long["Month"] = df_long["Month"].str.extract(r'R12 (\w+)', expand=False)

    message = """
    <div style='font-family: sans-serif; padding: 1rem;'>
    <div style='display: flex; align-items: flex-start; max-width: 80%; background-color: #fff8dc; border-radius: 10px; padding: 1rem;'>
        <div style='width: 30px; height: 30px; margin-right: 10px; background-color: #ffc107; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: bold;'>🤖</div>
        <div>
            <p>⚠️ <strong>While analyzing your sales data, we detected FEMA-confirmed disasters that likely impacted performance.</strong></p>
            <h4>🧾 Impact Summary:</h4>
    """

    for _, row in disaster_rows.iterrows():
        territory = row["Territory"]
        territory_name = row["Territory Name"]
        month = row["Month"]
        anomalous = row["Sales"]

        # Calculate average of other months from sales_df
        peer_sales = df_long[
            (df_long["Territory"] == territory) &
            (df_long["Month"] != month)
        ]["Sales"]

        avg_other = peer_sales.mean()
        drop_units = anomalous - avg_other
        drop_pct = (drop_units / avg_other * 100) if avg_other > 0 else None

        disaster = f"{row.get('IncidentType', 'Unknown')} — {row.get('DeclarationTitle', 'Unknown')}"
        begin = pd.to_datetime(row.get("IncidentBeginDate")).strftime("%B %d") if pd.notnull(row.get("IncidentBeginDate")) else "Unknown"
        end = pd.to_datetime(row.get("IncidentEndDate")).strftime("%d, %Y") if pd.notnull(row.get("IncidentEndDate")) else "Unknown"

        message += f"""
        <div style='margin-bottom: 1.2rem; padding-left: 1rem;'>
            <p><strong>📍 Territory:</strong> {territory_name}</p>
            <ul style='margin: 0.2rem 0 0.2rem 1rem;'>
                <li>📅 <strong>Month:</strong> {month}</li>
                <li>🎯 <strong>Anomalous Goal:</strong> {anomalous}</li>
                <li>📊 <strong>Average of Other Months:</strong> {avg_other:.1f}</li>
                <li>📉 <strong>Drop from Trend:</strong> {drop_units:.1f} units (⬇️ {drop_pct:.1f}%)</li>
                <li>🌪️ <strong>Disaster Impact:</strong> {disaster}</li>
                <li>🗓️ <strong>Disaster Period:</strong> {begin}–{end}</p></li>
            </ul>
        </div>
        """

        message += f"""
            <p style='margin-top: 1rem;'>📊 <em>We may exclude {month} from baseline calculations to ensure fair, achievable goals.</em></p>
            </div></div></div>
        """

    return message



In [2]:
sales_df=pd.read_excel("C:\IC Agents\crewai\icagents\Goal Setting sanitized Data.xlsx", sheet_name='Input Sales')
anomalies_df=detect_product1_anomalies_dynamic(sales_df,  -2.0)
anomalies_df


<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\deban\AppData\Local\Temp\ipykernel_14128\467481250.py:1: SyntaxWarning: invalid escape sequence '\I'
  sales_df=pd.read_excel("C:\IC Agents\crewai\icagents\Goal Setting sanitized Data.xlsx", sheet_name='Input Sales')


,Territory,Territory Name,Month,Sales,z_score
0,999999,Demo Unassigned,Jan,15,-2.677824
1,999999,Demo Unassigned,Feb,16,-3.069701
2,999999,Demo Unassigned,Mar,19,-2.495991
3,101004,Atlantic Zone,Apr,20,-2.166150
4,999999,Demo Unassigned,Apr,24,-2.017493
5,101003,Metro East,May,15,-2.470097
6,999999,Demo Unassigned,May,26,-2.102684
7,101002,Northeast Hub,Jun,10,-2.441288


In [ ]:
fema_df=pd.read_excel("C:\IC Agents\crewai\icagents\ZIP_to_Territory_with_FEMA_Data.xlsx",sheet_name="2025_ZIP_to_Territory")
cause_df=cross_verify_anomalies_with_fema(anomalies_df, fema_df)
#cause_df[(cause_df['TS Territory ID']==101002)]
#cause_df[cause_df['Disaster Match']==True]["Month"].unique().tolist()
cause_df=cause_df[cause_df['Disaster Match']==True][['Territory','Territory Name','Month','Sales','IncidentType','DeclarationTitle','IncidentBeginDate','IncidentEndDate']].drop_duplicates()
Group_df=cause_df.groupby(['Territory','Territory Name','Month','Sales','IncidentType','DeclarationTitle']).min('IncidentBeginDate').max('IncidentEndDate')

<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\deban\AppData\Local\Temp\ipykernel_14128\1943681543.py:1: SyntaxWarning: invalid escape sequence '\I'
  fema_df=pd.read_excel("C:\IC Agents\crewai\icagents\ZIP_to_Territory_with_FEMA_Data.xlsx",sheet_name="2025_ZIP_to_Territory")


,Territory,Territory Name,Month,Sales,IncidentType,DeclarationTitle,IncidentBeginDate,IncidentEndDate
3,101004,Atlantic Zone,Apr,20,Severe Storm,"SEVERE STORMS, STRAIGHT-LINE WINDS, TORNADOES,...",2025-04-02 00:00:00+00:00,2025-05-16T00:00:00.000Z
4,101004,Atlantic Zone,Apr,20,Severe Storm,"SEVERE STORMS, STRAIGHT-LINE WINDS, TORNADOES,...",2025-04-02 00:00:00+00:00,2025-05-16T00:00:00.000Z
6,101003,Metro East,May,15,Severe Storm,"SEVERE STORMS, STRAIGHT-LINE WINDS, AND TORNADOES",2025-05-16 00:00:00+00:00,2025-05-17T00:00:00.000Z
8,101002,Northeast Hub,Jun,10,Fire,ALDER SPRINGS FIRE,2025-06-16 00:00:00+00:00,NaN
9,101002,Northeast Hub,Jun,10,Fire,ROWENA FIRE,2025-06-11 00:00:00+00:00,NaN
10,101002,Northeast Hub,Jun,10,Flood,"SEVERE STORMS, FLOODING, AND LANDSLIDES",2025-06-23 00:00:00+00:00,NaN
11,101002,Northeast Hub,Jun,10,Fire,MARIE FIRE,2025-06-10 00:00:00+00:00,2025-06-12T00:00:00.000Z
12,101002,Northeast Hub,Jun,10,Fire,BEAR CREEK FIRE,2025-06-22 00:00:00+00:00,NaN
13,101002,Northeast Hub,Jun,10,Fire,NENANA RIDGE COMPLEX,2025-06-21 00:00:00+00:00,NaN
14,101002,Northeast Hub,Jun,10,Fire,COTTON 2 FIRE,2025-06-22 00:00:00+00:00,NaN


In [37]:
import pandas as pd
import numpy as np
def calculate_goals(national_goal: int, sales_file_path: str, min_growth: int, max_growth: int,product_weights: dict = None) -> pd.DataFrame:
    """
    Calculates IC goals using the national goal and input Excel data.
    
    Args:
        national_goal (int): The national goal as a whole number.
        sales_file_path (str): Path to the input Excel file.
        
    Returns:
        pd.DataFrame: A DataFrame containing the final goal calculation.
    """
    try:
        print(f"📌 National Goal received: {national_goal}")

        r12_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
        products = ["Product 1", "Product 2", "Product 3"]
        if product_weights is None:
            product_weights = {"Product 2": 60, "Product 3": 40}
            
        Region_name = {
            101: "East Division",
            201: "West Division",
            301: "Central Division"
        }
       

        # Load Excel data
        df_sales = pd.read_excel(sales_file_path, sheet_name='Input Sales')
        df_mapping = pd.read_excel(sales_file_path, sheet_name='1c. Inputs - Mappings', skiprows=5, usecols="C:D")
        df_mapping.drop_duplicates(inplace=True)

        # Merge mappings
        df = df_sales.merge(df_mapping, how="left", left_on="Territory", right_on="TS Territory")
        df.drop("TS Territory", axis=1, inplace=True)
        df["Region"] = df["Region"].astype(int)
        df["Region Name"] = df["Region"].map(Region_name)
        
        #print(df.head())
        

        # Calculate R12 totals
        for product in products:
            df[f"{product} R12 Total"] = df[[f"{product} R12 {month}" for month in r12_months]].sum(axis=1)

        df = df[["Region","Region Name", "Territory", "Territory Name"] + [f"{p} R12 Total" for p in products]]
        df['Baseline Goal'] = df[f'{products[0]} R12 Total'] / 2
        baseline_goal = df['Baseline Goal'].sum()
        
        for product in products[1:]:
            df[f"{product} Index"] = np.round(
                (df[f"{product} R12 Total"] / df[f"{product} R12 Total"].sum()) * 100, 1
            )

        df['Market Index'] = sum(df[f"{p} Index"] * product_weights[p] for p in products[1:])
        df['Market Index'] = np.round(df['Market Index'] / 100, 1)

        df["Growth/Potential Goal"] = np.round(abs(baseline_goal - national_goal) * df['Market Index'] / 100, 1)
        df["Preliminary Goal"] = df['Baseline Goal'] + df["Growth/Potential Goal"]
        df['Adjusted Preliminary Goal'] = np.round(
            (df["Preliminary Goal"] / df["Preliminary Goal"].sum()) * national_goal, 1
        )
        df['Growth%'] = (df['Adjusted Preliminary Goal'] / df['Baseline Goal'] - 1) * 100
       
        National_Growth=(df['Adjusted Preliminary Goal'].sum()/df['Baseline Goal'].sum())-1
        
        cap_min_terr_growth = National_Growth*min_growth*100
        cap_max_terr_growth = National_Growth*max_growth*100
        print(National_Growth, min_growth, max_growth)
        print(cap_min_terr_growth,cap_max_terr_growth)
        
        # Capping growth
        df['Capped Flag'] = np.where(
            df['Growth%'] < cap_min_terr_growth, 1,
            np.where(df['Growth%'] > cap_max_terr_growth, 2, 0)
        )
        df['Capped Growth%'] = df['Growth%'].clip(lower=cap_min_terr_growth, upper=cap_max_terr_growth)

        df['Initial Capped Goal'] = np.where(
            df['Baseline Goal'] == 0,
            df['Growth%'],
            (1 + df['Capped Growth%'] / 100) * df['Baseline Goal']
        )
        
        
        total_diff = max(0, df['Adjusted Preliminary Goal'].sum() - df['Initial Capped Goal'].sum())
        df['Spare growth'] = np.where(
            total_diff > 0,
            (cap_max_terr_growth / 100 - df['Capped Growth%'] / 100) * df['Baseline Goal'],
            (df['Capped Growth%'] / 100 - cap_min_terr_growth / 100) * df['Baseline Goal']
        )

        total_spare_growth = df['Spare growth'].sum()
        df["% of Nation"] = df['Spare growth'] / total_spare_growth * 100
        df["Goal Difference (Delta)"] = df["% of Nation"] * total_diff
        df["Final Goal (cartons)"] = np.round(df["Goal Difference (Delta)"] + df['Initial Capped Goal'], 0)
        #df.to_excel("output\calculated_goals.xlsx",index=False)
        return df

    except Exception as e:
        print(f"Error calculating goals: {str(e)}")
        raise
df=calculate_goals(5600, "C:\IC Agents\crewai\icagents\Goal Setting sanitized Data.xlsx",-1.0,1.0)
print(df.columns)
df

📌 National Goal received: 5600
0.011122144985104221 -1.0 1.0
-1.112214498510422 1.112214498510422
Index(['Region', 'Region Name', 'Territory', 'Territory Name',
       'Product 1 R12 Total', 'Product 2 R12 Total', 'Product 3 R12 Total',
       'Baseline Goal', 'Product 2 Index', 'Product 3 Index', 'Market Index',
       'Growth/Potential Goal', 'Preliminary Goal',
       'Adjusted Preliminary Goal', 'Growth%', 'Capped Flag', 'Capped Growth%',
       'Initial Capped Goal', 'Spare growth', '% of Nation',
       'Goal Difference (Delta)', 'Final Goal (cartons)'],
      dtype='object')


<>:104: SyntaxWarning: invalid escape sequence '\I'
<>:104: SyntaxWarning: invalid escape sequence '\I'
C:\Users\deban\AppData\Local\Temp\ipykernel_14128\1039330211.py:104: SyntaxWarning: invalid escape sequence '\I'
  df=calculate_goals(5600, "C:\IC Agents\crewai\icagents\Goal Setting sanitized Data.xlsx",-1.0,1.0)


,Region,Region Name,Territory,Territory Name,Product 1 R12 Total,Product 2 R12 Total,Product 3 R12 Total,Baseline Goal,Product 2 Index,Product 3 Index,...,Preliminary Goal,Adjusted Preliminary Goal,Growth%,Capped Flag,Capped Growth%,Initial Capped Goal,Spare growth,% of Nation,Goal Difference (Delta),Final Goal (cartons)
0,101,East Division,101001,East Coast,481,341,1361,240.5,2.6,3.1,...,242.2,242.2,0.706861,0,0.706861,242.200000,0.974876,11.829564,97.487587,340.0
1,101,East Division,101002,Northeast Hub,443,397,2066,221.5,3.1,4.7,...,223.8,223.8,1.038375,0,1.038375,223.800000,0.163555,1.984648,16.355511,240.0
2,101,East Division,101003,Metro East,456,208,968,228.0,1.6,2.2,...,229.1,229.1,0.482456,0,0.482456,229.100000,1.435849,17.423211,143.584906,373.0
3,101,East Division,101004,Atlantic Zone,641,788,3283,320.5,6.1,7.4,...,324.6,324.6,1.279251,2,1.112214,324.064647,0.000000,0.000000,0.000000,324.0
4,101,East Division,101005,Capital Region,602,503,1966,301.0,3.9,4.4,...,303.5,303.5,0.830565,0,0.830565,303.500000,0.847766,10.287153,84.776564,388.0
5,101,East Division,101009,Great Lakes,537,1043,4322,268.5,8.1,9.8,...,273.9,273.9,2.011173,2,1.112214,271.486296,0.000000,0.000000,0.000000,271.0
6,101,East Division,101010,Central Belt,722,701,3092,361.0,5.4,7.0,...,364.7,364.7,1.024931,0,1.024931,364.700000,0.315094,3.823490,31.509434,396.0
7,201,West Division,201001,Northwest Bay,467,435,1915,233.5,3.4,4.3,...,235.8,235.8,0.985011,0,0.985011,235.800000,0.297021,3.604179,29.702085,266.0
8,201,West Division,201002,NorCal Hub,631,1169,2608,315.5,9.1,5.9,...,320.3,320.3,1.521395,2,1.112214,319.009037,0.000000,0.000000,0.000000,319.0
9,201,West Division,201003,SoCal District,709,1110,2368,354.5,8.6,5.4,...,359.0,359.0,1.269394,2,1.112214,358.442800,0.000000,0.000000,0.000000,358.0
